In [1]:
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

with open('/content/drive/MyDrive/Dataset/Sentiment/Hotel_ABSA/Hotel_ABSA/Train.txt', 'r') as file:
    # Đọc nội dung của tệp và lưu vào biến content
    content = file.read()
    lines = content.split('\n\n')
    data = [line.split('\n') for line in lines]
    df_train = pd.DataFrame(data, columns=['id', 'text', 'label'])
    df_train = df_train.drop(['id'], axis=1)

with open('/content/drive/MyDrive/Dataset/Sentiment/Hotel_ABSA/Hotel_ABSA/Test.txt', 'r') as file:
    # Đọc nội dung của tệp và lưu vào biến content
    content = file.read()
    lines = content.split('\n\n')
    data = [line.split('\n') for line in lines]
    df_test = pd.DataFrame(data, columns=['id', 'text', 'label'])
    df_test = df_test.drop(['id'], axis=1)

with open('/content/drive/MyDrive/Dataset/Sentiment/Hotel_ABSA/Hotel_ABSA/Dev.txt', 'r') as file:
    # Đọc nội dung của tệp và lưu vào biến content
    content = file.read()
    lines = content.split('\n\n')
    data = [line.split('\n') for line in lines]
    df_val = pd.DataFrame(data, columns=['id', 'text', 'label'])
    df_val = df_val.drop(['id'], axis=1)

In [3]:
import re
import torch

# Define the mappings
aspect_to_index = {'facilities#cleanliness': 0, 'facilities#comfort': 1, 'facilities#design&features': 2, 'facilities#general': 3, 'facilities#miscellaneous': 4,
                   'facilities#prices': 5, 'facilities#quality': 6, 'food&drinks#miscellaneous': 7, 'food&drinks#prices': 8, 'food&drinks#quality': 9,
                   'food&drinks#style&options': 10, 'hotel#cleanliness': 11, 'hotel#comfort': 12, 'hotel#design&features': 13, 'hotel#general': 14,
                   'hotel#miscellaneous': 15, 'hotel#prices': 16, 'hotel#quality': 17, 'location#general': 18, 'rooms#cleanliness': 19, 'rooms#comfort': 20,
                   'rooms#design&features': 21, 'rooms#general': 22, 'rooms#miscellaneous': 23, 'rooms#prices': 24, 'rooms#quality': 25, 'room_amenities#cleanliness': 26,
                   'room_amenities#comfort': 27, 'room_amenities#design&features': 28, 'room_amenities#general': 29, 'room_amenities#miscellaneous': 30, 'room_amenities#prices': 31,
                   'room_amenities#quality': 32, 'service#general': 33}
# Adding 'None' to handle unspecified sentiments for any detected aspect
sentiment_to_index = {'positive': 1, 'negative': 0, 'neutral': 0, 'none': 0}

# Preprocess the labels
def preprocess_labels(label_str):
    labels = re.findall(r'\{(.*?)\}', label_str.lower())
    aspects = torch.zeros(len(aspect_to_index), dtype=torch.long)
    sentiments = torch.full((len(aspect_to_index),), sentiment_to_index['none'], dtype=torch.long)

    # Initialize with 'None'
    for label in labels:
        if ', ' in label:
            aspect, sentiment = label.split(', ')
            if aspect in aspect_to_index and sentiment in sentiment_to_index:  # Only process known aspects with sentiments
                idx = aspect_to_index[aspect]
                aspects[idx] = 1
                sentiments[idx] = sentiment_to_index[sentiment]

    return aspects, sentiments

df_train['aspects'], df_train['sentiments'] = zip(*df_train['label'].apply(preprocess_labels))
df_test['aspects'], df_test['sentiments'] = zip(*df_test['label'].apply(preprocess_labels))
df_val['aspects'], df_val['sentiments'] = zip(*df_val['label'].apply(preprocess_labels))

In [4]:
from bs4 import BeautifulSoup

for df in [df_train, df_test, df_val]:
  for sentence in range(0, len(df.text)):
    # xóa tag, link http
    processed_feature = BeautifulSoup(str(df.text[sentence]), "html.parser").get_text()
    #email-id
    processed_feature = re.sub('b[w-]+?@w+?.w{2,4}b', 'emailadd', processed_feature)
    #url
    processed_feature = re.sub('(http[s]?S+)|(w+.[A-Za-z]{2,4}S*)', 'urladd', processed_feature)

    # Xóa tất cả các ký tự đặc biệt
    processed_feature = re.sub(r'\W', ' ', processed_feature)

    # xóa từ có chứa chữ số
    # processed_feature = re.sub('W*dw*', '', processed_feature)

    # Chuyển đổi sang chữ thường
    processed_feature = processed_feature.lower()

    # Thay thế nhiều khoảng trắng bằng một khoảng trắng
    processed_feature = re.sub(r'\s+', ' ', processed_feature, flags=re.I)

    # processed_features.append(processed_feature)
    df.text[sentence] = processed_feature

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df.text[sentence] = processed_feature
<ipython-input-4-11490973c74e>:25: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/

In [5]:
df_train['aspects'] = df_train['aspects'].apply(lambda x: x.numpy())
df_train['sentiments'] = df_train['sentiments'].apply(lambda x: x.numpy())

df_test['aspects'] = df_test['aspects'].apply(lambda x: x.numpy())
df_test['sentiments'] = df_test['sentiments'].apply(lambda x: x.numpy())

df_val['aspects'] = df_val['aspects'].apply(lambda x: x.numpy())
df_val['sentiments'] = df_val['sentiments'].apply(lambda x: x.numpy())

In [6]:
df_train

,text,label,aspects,sentiments
0,vừa qua tôi có dùng dịch vụ tại khách sạn ttc ...,"{HOTEL#GENERAL, positive}, {SERVICE#GENERAL, p...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ..."
1,tuy nhiên buffet sáng ở đây không được ngon và...,"{FOOD&DRINKS#QUALITY, negative}, {FOOD&DRINKS#...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,nhìn chung dịch vụ khách sạn cũng tốt chỉ có đ...,"{FOOD&DRINKS#QUALITY, negative}, {SERVICE#GENE...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,đội ngũ nhân viên phục vụ chu đáo tận tình,"{SERVICE#GENERAL, positive}","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,trang thiết bị nội thất hài hoà tiện nghi hiện...,"{ROOM_AMENITIES#DESIGN&FEATURES, positive}, {R...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...
7175,dù thái độ phục vụ bình thường nhưng phòng ở t...,"{SERVICE#GENERAL, neutral}, {ROOMS#DESIGN&FEAT...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
7176,dịch vụ ở khách sạn mường thanh grand đà nẵng ...,"{SERVICE#GENERAL, positive}, {ROOMS#CLEANLINES...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
7177,nhân viên phục vụ niềm nở vị trí đi lại thuận ...,"{SERVICE#GENERAL, positive}, {LOCATION#GENERAL...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
7178,khách sạn mường thanh đà nẵng tốt thôi chẳng c...,"{HOTEL#COMFORT, positive}","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."


In [7]:
import gensim

documents = [_text.split() for _text in df_train.text]
w2v_model = gensim.models.word2vec.Word2Vec(vector_size=128,
                                            window=7,
                                            min_count=10,
                                            workers=8)

w2v_model.build_vocab(documents)

w2v_model.train(documents, total_examples=len(documents), epochs=32)

(2861352, 4184960)

In [8]:
words = w2v_model.wv
vocab_size = len(words)
print("Vocab size", vocab_size)

Vocab size 758


In [9]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer()
tokenizer.fit_on_texts(df_train.text)

vocab_size = len(tokenizer.word_index) + 1
print("Total words", vocab_size)

Total words 2197


In [10]:
embedding_matrix = np.zeros((vocab_size, 128))
for word, i in tokenizer.word_index.items():
  if word in w2v_model.wv:
    embedding_matrix[i] = w2v_model.wv[word]
print(embedding_matrix.shape)

(2197, 128)


In [11]:
X_train = pad_sequences(tokenizer.texts_to_sequences(df_train.text.values), maxlen=128)
X_test = pad_sequences(tokenizer.texts_to_sequences(df_test.text.values), maxlen=128)
X_val = pad_sequences(tokenizer.texts_to_sequences(df_val.text.values), maxlen=128)

In [12]:
aspect_train = np.stack(df_train.aspects.values)
aspect_test = np.stack(df_test.aspects.values)
aspect_val = np.stack(df_val.aspects.values)

sentiment_train = np.stack(df_train.sentiments.values)
sentiment_test = np.stack(df_test.sentiments.values)
sentiment_val = np.stack(df_val.sentiments.values)

In [13]:
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import BatchNormalization, Conv1D, MaxPooling1D, Activation, Flatten, Dropout, Dense, DepthwiseConv1D, GlobalAveragePooling1D, Input, Embedding

import tensorflow.keras.backend as K

INPUT_SHAPE = (X_train.shape[1],)
N_ASPECT = 34
N_SENTIMENT = 34

class AspectSentimentNet(object):
  def _aspect_basenetwork(inputs, classes = N_ASPECT, finAct = 'softmax'):
    embedding = Embedding(input_dim=vocab_size,
                      output_dim=128,
                      weights=[embedding_matrix],
                      trainable=True)(inputs)
    # CONV => RELU => POOL
    x = Conv1D(filters=128, kernel_size=3, padding='same', activation='relu')(embedding)
    x = BatchNormalization(axis=-1)(x)
    x = MaxPooling1D(pool_size=3)(x)
    x = Dropout(0.3)(x)

    # CONV => RELU => POOL
    x = Conv1D(filters=128, kernel_size=3, padding='same', activation='relu')(x)
    x = BatchNormalization(axis=-1)(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    # CONV => RELU => POOL
    x = Conv1D(filters=128, kernel_size=3, padding='same', activation='relu')(x)
    x = BatchNormalization(axis=-1)(x)
    x = MaxPooling1D(pool_size=2)(x)

    # MLP classifier
    x = Flatten()(x)
    x = Dense(128, activation='relu')(x)
    x = BatchNormalization(axis=-1)(x)
    x = Dense(classes)(x)
    x = Activation(finAct, name="aspect_output")(x)

    return x

  def _sentiment_basenetwork(inputs, classes = N_SENTIMENT, finAct = 'softmax'):
    embedding = Embedding(input_dim=vocab_size,
                          output_dim=128,
                          weights=[embedding_matrix],
                          trainable=True)(inputs)
    # CONV => RELU => POOL
    x = Conv1D(filters=128, kernel_size=3, padding='same', activation='relu')(embedding)
    x = BatchNormalization(axis=-1)(x)
    x = MaxPooling1D(pool_size=3)(x)
    x = Dropout(0.3)(x)

    # CONV => RELU => POOL
    x = Conv1D(filters=128, kernel_size=3, padding='same', activation='relu')(x)
    x = BatchNormalization(axis=-1)(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    # CONV => RELU => POOL
    x = Conv1D(filters=128, kernel_size=3, padding='same', activation='relu')(x)
    x = BatchNormalization(axis=-1)(x)
    x = MaxPooling1D(pool_size=2)(x)

    # MLP classifier
    x = Flatten()(x)
    x = Dense(128, activation='relu')(x)
    x = BatchNormalization(axis=-1)(x)
    x = Dense(classes)(x)
    x = Activation(finAct, name="sentiment_output")(x)

    return x

  @staticmethod
  def build(inputShape=INPUT_SHAPE, numAspect = N_ASPECT, numSentiment = N_SENTIMENT):
    inputs = Input(shape=INPUT_SHAPE)
    aspectBranch=AspectSentimentNet._aspect_basenetwork(inputs=inputs,
      classes = numAspect, finAct='softmax')
    sentimentBranch=AspectSentimentNet._sentiment_basenetwork(inputs=inputs,
      classes = numSentiment, finAct='softmax')

    # Tạo một mô hình sử dụng đầu vào là chuỗi embedding, sau đó mô hình sẽ rẽ nhánh, một nhánh xác định đặc trưng của aspect và một nhánh xác định đặc trưng của sentiment
    model = Model(inputs=inputs,
                  outputs=[aspectBranch, sentimentBranch],
                  name="aspect_sentiment_net")

    return model

model = AspectSentimentNet.build(inputShape=INPUT_SHAPE, numAspect = N_ASPECT, numSentiment = N_SENTIMENT)


In [14]:
from tensorflow.keras.optimizers import AdamW

LR_RATE = 5e-5
EPOCHS = 100
opt = AdamW(learning_rate=LR_RATE, weight_decay=LR_RATE / EPOCHS)

losses = {"aspect_output": "binary_crossentropy",
          "sentiment_output": "binary_crossentropy"}
metrics = {"aspect_output": "accuracy",
           "sentiment_output": "accuracy"}
model.compile(loss=losses, optimizer=opt, metrics=metrics)


In [15]:
history = model.fit(X_train, {"aspect_output": aspect_train, "sentiment_output": sentiment_train},
                    validation_data=(X_val, {"aspect_output": aspect_val, "sentiment_output": sentiment_val}),
                    batch_size=32,
                    steps_per_epoch=len(X_train) // 32,
                    epochs=EPOCHS, verbose=1)

Epoch 1/100
224/224 ━━━━━━━━━━━━━━━━━━━━ 21s 20ms/step - aspect_output_accuracy: 0.0404 - aspect_output_loss: 0.8277 - loss: 1.6549 - sentiment_output_accuracy: 0.0247 - sentiment_output_loss: 0.8272 - val_aspect_output_accuracy: 0.0654 - val_aspect_output_loss: 0.6903 - val_loss: 1.4127 - val_sentiment_output_accuracy: 0.0403 - val_sentiment_output_loss: 0.7223
Epoch 2/100
224/224 ━━━━━━━━━━━━━━━━━━━━ 8s 836us/step - aspect_output_accuracy: 0.0000e+00 - aspect_output_loss: 0.7636 - loss: 1.5172 - sentiment_output_accuracy: 0.0000e+00 - sentiment_output_loss: 0.7537 - val_aspect_output_accuracy: 0.0667 - val_aspect_output_loss: 0.6903 - val_loss: 1.4129 - val_sentiment_output_accuracy: 0.0415 - val_sentiment_output_loss: 0.7225
Epoch 3/100


/usr/lib/python3.10/contextlib.py:153: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(typ, value, traceback)


224/224 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - aspect_output_accuracy: 0.0770 - aspect_output_loss: 0.7375 - loss: 1.4764 - sentiment_output_accuracy: 0.0454 - sentiment_output_loss: 0.7389 - val_aspect_output_accuracy: 0.1610 - val_aspect_output_loss: 0.6779 - val_loss: 1.3615 - val_sentiment_output_accuracy: 0.0579 - val_sentiment_output_loss: 0.6836
Epoch 4/100
224/224 ━━━━━━━━━━━━━━━━━━━━ 0s 618us/step - aspect_output_accuracy: 0.0833 - aspect_output_loss: 0.7005 - loss: 1.3890 - sentiment_output_accuracy: 0.0000e+00 - sentiment_output_loss: 0.6885 - val_aspect_output_accuracy: 0.1610 - val_aspect_output_loss: 0.6777 - val_loss: 1.3608 - val_sentiment_output_accuracy: 0.0591 - val_sentiment_output_loss: 0.6831
Epoch 5/100
224/224 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - aspect_output_accuracy: 0.1157 - aspect_output_loss: 0.6803 - loss: 1.3645 - sentiment_output_accuracy: 0.0616 - sentiment_output_loss: 0.6842 - val_aspect_output_accuracy: 0.2277 - val_aspect_output_loss: 0.6485 - val_loss

In [16]:
sentence = "Nhà hàng có không gian đẹp, nhưng chất lượng món ăn lại không như mong đợi, phục vụ hơi chậm."
input = pad_sequences(tokenizer.texts_to_sequences([sentence]), maxlen=128)
prediction = model.predict(input)
print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
[array([[4.98251198e-03, 1.03673176e-03, 6.42277580e-03, 1.61036127e-03,
        5.90619305e-03, 1.18757074e-03, 7.69957341e-03, 1.16300778e-02,
        7.13361055e-03, 8.86067003e-02, 8.29831585e-02, 1.77335390e-03,
        4.41619987e-03, 1.76860057e-02, 5.20095043e-03, 8.40341579e-03,
        5.80139924e-04, 6.07271492e-03, 5.93874836e-04, 1.43668381e-03,
        2.27089110e-03, 6.96780952e-03, 3.24315182e-03, 2.32610642e-03,
        7.80944247e-04, 1.08847525e-02, 9.63527535e-04, 8.25965544e-04,
        2.07659719e-03, 8.36108811e-04, 3.65378289e-03, 2.89253599e-04,
        4.21977835e-03, 6.95298731e-01]], dtype=float32), array([[0.00427187, 0.00394287, 0.02817   , 0.00571762, 0.00353088,
        0.00624875, 0.00578989, 0.00598271, 0.02406389, 0.1713935 ,
        0.15972222, 0.01643038, 0.02437281, 0.03767999, 0.16842736,
        0.01109789, 0.0096165 , 0.01384919, 0.00262855, 0.00761722,
        0.00866812, 0.0114053 , 0.0180626 , 0.00235448, 0

In [17]:
score = model.evaluate(X_test, {"aspect_output": aspect_test, "sentiment_output": sentiment_test}, batch_size=32)


64/64 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - aspect_output_accuracy: 0.5769 - aspect_output_loss: 0.0771 - loss: 0.1347 - sentiment_output_accuracy: 0.4287 - sentiment_output_loss: 0.0576


In [19]:
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluation(aspect_test, sentiment_test):
  y_pred = model.predict(X_test)

  aspect_pred_binary = (y_pred[0] > 0.5).astype(int)

  precision_aspect = precision_score(aspect_test, aspect_pred_binary, average='micro')
  recall_aspect = recall_score(aspect_test, aspect_pred_binary, average='micro')
  f1_aspect = f1_score(aspect_test, aspect_pred_binary, average='micro')

  sentiment_pred_class = (y_pred[0] > 0.5).astype(int)
  precision_sentiment = precision_score(sentiment_test,sentiment_pred_class, average='micro')
  recall_sentiment = recall_score(sentiment_test,sentiment_pred_class, average='micro')
  f1_sentiment = f1_score(sentiment_test,sentiment_pred_class, average='micro')

  precision = (precision_aspect + precision_sentiment) / 2
  recall = (recall_aspect + recall_sentiment) / 2
  f1 = (f1_aspect + f1_sentiment) / 2

  print("Aspect, Sentiment and Total respectively")
  print("Precision:", precision_aspect, precision_sentiment, precision)
  print("Recall:", recall_aspect, recall_sentiment, recall)
  print("F1:", f1_aspect, f1_sentiment, f1)

evaluation(aspect_test, sentiment_test)

64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Aspect, Sentiment and Total respectively
Precision: 0.8506988564167726 0.7090216010165185 0.7798602287166455
Recall: 0.4107361963190184 0.45329000812347686 0.43201310222124767
F1: 0.5539925527513446 0.5530227948463825 0.5535076737988636
